# LLM Steganography Benchmarks (Colab)

Jeden notebook = jeden wybrany test + jeden model + jeden lub wszystkie progi.

Dostępne testy (`benchmarks.md`):
- `humaneval` — Pass@1 (zdolność rozumowania / kod)
- `perplexity` — naturalność tekstu (PPL)
- `capacity` — pojemność steganograficzna (średnia liczba kandydatów, BPT, ER)
- `binoculars` — niewykrywalność (Binoculars score, AI detection rate)

Wyniki trafiają do:
- `results/<timestamp>_<model>_<test>_thX>/` — pełny przebieg (config, summary, samples)
- `results/summary.csv` — zbiorcza tabela wszystkich przebiegów

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
PROJECT_DIR = '/content/drive/MyDrive/steganography_LLM'
%cd $PROJECT_DIR

In [ ]:
!pip install -U pip
!pip install -r requirements.txt

In [ ]:
import os
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = hf_token
login(token=hf_token, add_to_git_credential=False)
print('HF login OK')

## Konfiguracja przebiegu

Zmień tylko te 3 zmienne, potem uruchom komórkę poniżej.

In [ ]:
from pathlib import Path

required = ['run_experiments.py', 'dummy_processor.py']
missing = [name for name in required if not Path(name).exists()]
if missing:
    raise FileNotFoundError(f'Brakuje plikow na Drive: {missing}')
print('Wymagane pliki OK')

In [ ]:
# Test: humaneval | perplexity | capacity | binoculars
TEST = 'humaneval'

# Model: qwen | llama | gemma (albo pełne HF model id)
MODEL = 'qwen'

# Threshold: np. 0.01 albo 'all' (0.0, 0.01, 0.05, 0.1)
THRESHOLD = 0.01

# Opcjonalnie
TOP_N = 15
MAX_NEW_TOKENS = 256

In [ ]:
!python run_experiments.py \
  --test {TEST} \
  --model {MODEL} \
  --threshold {THRESHOLD} \
  --top-n {TOP_N} \
  --max-new-tokens {MAX_NEW_TOKENS}

In [ ]:
import csv
from pathlib import Path

summary_csv = Path('results/summary.csv')
if summary_csv.exists():
    with summary_csv.open(newline='', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    print(f'Rows in summary.csv: {len(rows)}')
    for row in rows[-5:]:
        print(row)
else:
    print('summary.csv not found yet')